# Probe training on merged shards

Same logic as `train.ipynb`'s probe-training section. Only difference: `X` and `y` are loaded from the merged shards (output of `shard_merge.ipynb`) instead of being produced in-memory by a generation loop.

Runs on CPU in a few minutes. No GPU, no compute units.

In [ ]:
# === Colab bootstrap ===
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/cs639-outputs"
else:
    OUTPUT_DIR = "."

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"IN_COLAB={IN_COLAB}  OUTPUT_DIR={OUTPUT_DIR}")

In [ ]:
%pip install -q torch pandas

## Config

In [ ]:
# --- Inputs (from shard_merge.ipynb) ---
X_PATH     = os.path.join(OUTPUT_DIR, "X.pt")
Y_PATH     = os.path.join(OUTPUT_DIR, "y.pt")

# --- Output ---
PROBE_PATH = os.path.join(OUTPUT_DIR, "linear_probe.pt")

# --- Hyperparameters (identical to train.ipynb) ---
HIDDEN_DIM   = 3584   # DeepSeek-R1-Distill-Qwen-7B
PROBE_EPOCHS = 30
PROBE_LR     = 5e-4
BATCH_SIZE   = 256

In [ ]:
# === Load merged X, y ===
import torch

assert os.path.exists(X_PATH), f"Missing {X_PATH} — run shard_merge.ipynb first"

X = torch.load(X_PATH, map_location="cpu", weights_only=False)
y = torch.load(Y_PATH, map_location="cpu", weights_only=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"X: {tuple(X.shape)}  y: {tuple(y.shape)}  device: {device}")
print(f"Label distribution: {int(y.sum())} correct / {int((1 - y).sum())} incorrect")

In [ ]:
# === Build dataset + loader (same as train.ipynb) ===
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# === Probe, loss, optimizer (same as train.ipynb) ===
import torch.nn as nn

class LinearProbe(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc(x).squeeze(-1)

probe     = LinearProbe(HIDDEN_DIM).to(device)
optimizer = torch.optim.Adam(probe.parameters(), lr=PROBE_LR)
criterion = nn.BCEWithLogitsLoss()
print(probe)

In [ ]:
# === Training loop (same as train.ipynb) ===
for epoch in range(1, PROBE_EPOCHS + 1):
    probe.train()
    total_loss, n_correct, n_total = 0.0, 0, 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = probe(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)
        n_correct  += ((logits > 0).long() == yb.long()).sum().item()
        n_total    += len(yb)

    print(f"Epoch {epoch:3d}/{PROBE_EPOCHS}  "
          f"loss={total_loss / n_total:.4f}  "
          f"acc={n_correct / n_total:.4f}")

In [ ]:
# === Save ===
torch.save(probe.state_dict(), PROBE_PATH)
print(f"Saved probe -> {PROBE_PATH}")